In [24]:
import json
import time
import os
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()
# 1. Fetch key from environment variables
api_key_env = os.environ.get("OPENROUTER_API_KEY")


client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key_env,
)


# Model: Qwen 3.5 9B Instruct
MODEL_ID = "qwen/qwen3.5-9b"

In [13]:
def get_model_response(prompt):
    """
    Sends request to OpenRouter and cleans the output of any 'Thinking' text.
    """
    system_prompt = (
        "You are a helpful assistant. Follow instructions strictly. "
        "Provide your answer in plain text ONLY unless states otherwise. "

    )

    max_retries = 5
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0, # Mandatory for research consistency
                extra_headers={
                    "HTTP-Referer": "http://localhost",
                    "X-Title": "ANOVA Prompt Syntax Study",
                }
            )

            raw_content = completion.choices[0].message.content.strip()

            # Post-processing: Remove accidental 'Thinking Process' if it appears
            # This cleans up headers like "Thinking Process:", "Thought:", etc.
            cleaned_content = re.sub(r"(?i)(Thinking Process|Thought|Analysis|Reasoning|Action):.*?\n\n", "", raw_content, flags=re.DOTALL)
            return cleaned_content.strip()

        except Exception as e:
            wait_time = (2 ** attempt)
            if attempt < max_retries - 1:
                time.sleep(wait_time)
            else:
                return f"API_ERROR: {str(e)}"


In [25]:
# Create a lock to prevent multiple threads from writing to the file at the same time
file_lock = threading.Lock()

def process_single_entry(entry):
    """ Helper to process all variations of one prompt entry """
    res_entry = {
        "key": entry['key'],
        "instruction_id_list": entry['instruction_id_list'],
        "kwargs": entry['kwargs'],
        "responses": {}
    }

    # We could parallelize inside here too, but parallelizing entries is simpler
    for fmt, prompt_text in entry['variations'].items():
        if prompt_text:
            res_entry['responses'][fmt] = get_model_response(prompt_text)

    return res_entry

def process_anova_batch_parallel(input_path, output_path, max_workers=10):
    if not os.path.exists(input_path):
        print(f"Error: Could not find {input_path}")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        master_data = json.load(f)

    queue = [e for e in master_data if str(e.get('status','')).lower() == 'done']
    results = []

    # Resume Logic
    if os.path.exists(output_path):
        try:
            with open(output_path, 'r', encoding='utf-8') as f:
                results = json.load(f)
                processed_keys = {str(r['key']) for r in results}
                queue = [e for e in queue if str(e['key']) not in processed_keys]
                print(f"Resume: {len(results)} items loaded. {len(queue)} left.")
        except Exception:
            pass

    print(f"Starting Parallel API processing with {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all entries to the thread pool
        future_to_entry = {executor.submit(process_single_entry, entry): entry for entry in queue}

        # Use tqdm to track progress
        for future in tqdm(as_completed(future_to_entry), total=len(queue), desc="ANOVA Progress"):
            try:
                res_entry = future.result()

                # Use lock to update shared list and save to file safely
                with file_lock:
                    results.append(res_entry)
                    with open(output_path, 'w', encoding='utf-8') as f:
                        json.dump(results, f, indent=2)
            except Exception as e:
                print(f"\nCritical error processing entry: {e}")

    print(f"\nProcessing Complete. Saved to: {output_path}")


In [19]:
test_prompt = "Explain why XML tags help structured prompts in exactly one sentence."
print("Testing API Response...")
start = time.time()
response = get_model_response(test_prompt)
print(f"Response: {response}")
print(f"Time: {time.time() - start:.2f}s")

Testing API Response...
Response: XML tags help structured prompts by organizing information into distinct sections that reduce ambiguity and enable the model to distinguish between instructions and content more effectively.
Time: 30.20s


In [ ]:
# Ensure these files are in your PyCharm project folder
INPUT_JSON = 'data/IfBenchTestFile.json'
OUTPUT_JSON = 'results/IfBenchResults.json'



# max_workers=10 means 10 entries (50 variations) can be processed at the same time.
# Start with 10; increase to 20 if your OpenRouter rate limit allows it.
process_anova_batch_parallel(INPUT_JSON, OUTPUT_JSON, max_workers=10)

Starting Parallel API processing with 10 workers...


ANOVA Progress:   0%|          | 0/300 [00:00<?, ?it/s]